# 🗺️ Matchify: Neshan Maps (neshan.org) Venue Matcher

A high-speed, rate-limit immune asynchronous pipeline for matching venue databases against **Neshan Maps**.

### 🌟 Extracted Fields:
- **Canonical Identifier:** `crowd_id` (e.g. `abfcf1e7fcbbd5de02e2e324a2c0bb88`), `neshan_url`
- **Verified Location:** Exact latitude and longitude on Neshan
- **Rich Metadata:** `title`, `category_name`, `address` (street & district), `phone`, `rating`, `votes_count`, `status_text`
- **Scoring & Verification:** `match_tier` (`strict` ~99% vs `loose` review queue), `name_score`, `distance_meters`, `composite_score`, `phone_matched`

### 🚀 Quickstart on Kaggle:
1. Set **Internet -> Internet On** in the right panel (Notebook Options).
2. Attach your database shard via **+ Add Input** (e.g. `cafochino_shard_01...db`).
3. Click **Save Version -> Save & Run All (Commit)** or run cells interactively.

In [ ]:
# Step 1: Install lightweight dependencies
!pip install -q httpx loguru tqdm

In [ ]:
# Step 2: Self-contained core engine (Normalizer, Matcher, NeshanClient, DatabaseManager)
import asyncio
import base64
from dataclasses import dataclass, field
from datetime import datetime, timezone
from difflib import SequenceMatcher
from enum import Enum
import glob
import json
import math
import os
from pathlib import Path
import random
import re
import secrets
import sqlite3
import time
from typing import List, Optional, Dict, Any, Tuple
import urllib.parse

import httpx
from loguru import logger
from tqdm import tqdm

STOP_WORDS = {
    'کافه', 'رستوران', 'کافی', 'شاپ', 'قهوه', 'بار', 'لانژ', 'فست', 'فود', 'عمارت', 'خانه', 'سرای',
    'cafe', 'café', 'restaurant', 'coffee', 'shop', 'bar', 'lounge', 'fast', 'food', 'bakery',
    'شعبه', 'ایرانی', 'سنتی', 'مدرن', 'اصلی', 'مرکزی', 'بین', 'الملل', 'بین‌المللی',
    'تهران', 'اصفهان', 'مشهد', 'شیراز', 'تبریز', 'اهواز', 'کرج', 'قم', 'کیش'
}

PERSIAN_TO_LATIN = {
    'آ': 'a', 'ا': 'a', 'ب': 'b', 'پ': 'p', 'ت': 't', 'ث': 's', 'ج': 'j', 'چ': 'ch', 'ح': 'h',
    'خ': 'kh', 'د': 'd', 'ذ': 'z', 'ر': 'r', 'ز': 'z', 'ژ': 'zh', 'س': 's', 'ش': 'sh', 'ص': 's',
    'ض': 'z', 'ط': 't', 'ظ': 'z', 'ع': 'a', 'غ': 'gh', 'ف': 'f', 'ق': 'gh', 'ک': 'k', 'گ': 'g',
    'ل': 'l', 'م': 'm', 'ن': 'n', 'و': 'v', 'ه': 'h', 'ی': 'i', 'ي': 'i', 'ك': 'k', 'ة': 'h'
}

COMMON_TRANSLATIONS = {'مستر': 'mr', 'بیک': 'bake', 'نیلی': 'nili', 'پروچلو': 'prochelo', 'ویونا': 'viuna', 'لامیز': 'lameiz'}

def normalize_text(text: str) -> str:
    if not text:
        return ''
    t = text.lower()
    t = re.sub(r'[\u200c\u200b\u200e\u200f\u202a-\u202e]', ' ', t)
    t = t.replace('\u0640', '')
    t = t.replace('ي', 'ی').replace('ى', 'ی').replace('ك', 'ک').replace('ة', 'ه')
    t = t.replace('آ', 'ا').replace('أ', 'ا').replace('إ', 'ا').replace('ؤ', 'و').replace('ئ', 'ی')
    t = re.sub(r'[\u064B-\u065F\u0670]', '', t)
    for k, v in {'۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9'}.items():
        t = t.replace(k, v)
    t = re.sub(r'[\(\)\[\]\{\}\-_,\.\+:;/\&\\|\'\"`!?؟،؛«»•–—…]', ' ', t)
    return ' '.join(t.split())

def clean_tokens(text: str) -> List[str]:
    norm = normalize_text(text)
    tokens = norm.split()
    f = [w for w in tokens if w not in STOP_WORDS]
    return f if f else tokens

def to_phonetic_latin(tokens: List[str]) -> List[str]:
    latin = []
    for tok in tokens:
        if tok in COMMON_TRANSLATIONS:
            latin.append(COMMON_TRANSLATIONS[tok])
            continue
        if all(ord(c) < 128 for c in tok):
            latin.append(tok)
            continue
        res = ''.join(PERSIAN_TO_LATIN.get(c, c) for c in tok)
        res = re.sub(r'[^a-z0-9]', '', res)
        res = re.sub(r'(.)\1+', r'\1', res)
        if res:
            latin.append(res)
    return latin

def normalize_phone(phone: str) -> str:
    if not phone:
        return ''
    p = phone
    for k, v in {'۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9'}.items():
        p = p.replace(k, v)
    digits = re.sub(r'\D', '', p)
    if digits.startswith('98'):
        digits = digits[2:]
    elif digits.startswith('0'):
        digits = digits[1:]
    return digits

def check_phone_match(phone1: Optional[str], phone2: Optional[str]) -> bool:
    p1 = normalize_phone(phone1 or '')
    p2 = normalize_phone(phone2 or '')
    if len(p1) >= 7 and len(p2) >= 7:
        return (p1 == p2) or p1.endswith(p2[-7:]) or p2.endswith(p1[-7:])
    return False

def compute_name_similarity(name1: str, name2: str) -> float:
    if not name1 or not name2:
        return 0.0
    tok1 = clean_tokens(name1)
    tok2 = clean_tokens(name2)
    s1, s2 = ' '.join(tok1), ' '.join(tok2)
    direct = SequenceMatcher(None, s1, s2).ratio()
    joined1, joined2 = ''.join(tok1), ''.join(tok2)
    joined = SequenceMatcher(None, joined1, joined2).ratio()
    lat1, lat2 = to_phonetic_latin(tok1), to_phonetic_latin(tok2)
    phonetic = SequenceMatcher(None, ''.join(lat1), ''.join(lat2)).ratio() if (lat1 and lat2) else 0.0
    set1 = {t for t in lat1 if len(t) >= 3}
    set2 = {t for t in lat2 if len(t) >= 3}
    if set1 and set2:
        overlap = len(set1 & set2) / min(len(set1), len(set2))
        containment = 0.85 if (set1.issubset(set2) or set2.issubset(set1)) else (overlap * 0.8)
    else:
        containment = 0.0
    return round(max(direct, joined, phonetic, containment), 3)

def haversine_distance(lat1: Optional[float], lon1: Optional[float], lat2: Optional[float], lon2: Optional[float]) -> float:
    if None in (lat1, lon1, lat2, lon2):
        return float('inf')
    try:
        lat1, lon1, lat2, lon2 = float(lat1), float(lon1), float(lat2), float(lon2)
        if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90 and -180 <= lon1 <= 180 and -180 <= lon2 <= 180):
            return float('inf')
        if (lat1 == 0 and lon1 == 0) or (lat2 == 0 and lon2 == 0):
            return float('inf')
    except Exception:
        return float('inf')
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(max(0.0, 1.0 - a)))
    return round(R * c, 1)

class MatchTier(str, Enum):
    STRICT = 'strict'
    LOOSE = 'loose'
    REJECTED = 'none'

@dataclass
class MatchCandidate:
    candidate_id: str
    title: str
    latitude: Optional[float]
    longitude: Optional[float]
    address: Optional[str] = None
    phone: Optional[str] = None
    category: Optional[str] = None
    url: Optional[str] = None
    raw_data: Optional[Dict[str, Any]] = field(default_factory=dict)

@dataclass
class MatchResult:
    is_match: bool
    match_tier: MatchTier
    match_rule: str
    composite_score: float
    name_score: float
    distance_meters: float
    phone_matched: bool
    candidate: Optional[MatchCandidate] = None
    rejection_reason: Optional[str] = None

def evaluate_candidate(target_name: str, target_lat: Optional[float], target_lng: Optional[float], target_phone: Optional[str], candidate: MatchCandidate) -> MatchResult:
    name_score = compute_name_similarity(target_name, candidate.title)
    dist = haversine_distance(target_lat, target_lng, candidate.latitude, candidate.longitude)
    phone_matched = check_phone_match(target_phone, candidate.phone)
    dist_score = 1.0 if dist <= 30 else max(0.0, 1.0 - (dist - 30) / 970.0) if dist < 1000 else 0.0
    composite = round((0.65 * name_score) + (0.35 * dist_score) + (0.15 if phone_matched else 0.0), 3)

    # Strict Rules
    if phone_matched and name_score >= 0.45 and dist <= 2500:
        return MatchResult(True, MatchTier.STRICT, 'STRICT_PHONE_AND_NAME_MATCH', composite, name_score, dist, True, candidate)
    if name_score >= 0.85 and dist <= 150:
        return MatchResult(True, MatchTier.STRICT, 'STRICT_HIGH_SIMILARITY_CLOSE_PROXIMITY', composite, name_score, dist, phone_matched, candidate)
    if name_score >= 0.95 and dist <= 350:
        return MatchResult(True, MatchTier.STRICT, 'STRICT_EXACT_NAME_MODERATE_RADIUS', composite, name_score, dist, phone_matched, candidate)

    # Loose Rules
    if name_score >= 0.60 and dist <= 500:
        return MatchResult(True, MatchTier.LOOSE, 'LOOSE_MODERATE_SIMILARITY_NEARBY', composite, name_score, dist, phone_matched, candidate)
    if name_score >= 0.80 and dist <= 1500:
        return MatchResult(True, MatchTier.LOOSE, 'LOOSE_HIGH_SIMILARITY_EXTENDED_RADIUS', composite, name_score, dist, phone_matched, candidate)
    if name_score >= 0.50 and dist <= 80:
        return MatchResult(True, MatchTier.LOOSE, 'LOOSE_CLOSE_PROXIMITY_PARTIAL_NAME', composite, name_score, dist, phone_matched, candidate)

    return MatchResult(False, MatchTier.REJECTED, 'REJECTED', composite, name_score, dist, phone_matched, candidate, f'Distance {dist}m / Name score {name_score}')

def rank_and_select_best_match(target_name: str, target_lat: Optional[float], target_lng: Optional[float], target_phone: Optional[str], candidates: List[MatchCandidate]) -> MatchResult:
    if not candidates:
        return MatchResult(False, MatchTier.REJECTED, 'NO_CANDIDATES', 0.0, 0.0, float('inf'), False, rejection_reason='No search results')
    evaluated = [evaluate_candidate(target_name, target_lat, target_lng, target_phone, c) for c in candidates]
    stricts = [r for r in evaluated if r.match_tier == MatchTier.STRICT]
    if stricts:
        stricts.sort(key=lambda r: (r.phone_matched, r.composite_score, -r.distance_meters), reverse=True)
        return stricts[0]
    looses = [r for r in evaluated if r.match_tier == MatchTier.LOOSE]
    if looses:
        looses.sort(key=lambda r: (r.composite_score, -r.distance_meters), reverse=True)
        return looses[0]
    evaluated.sort(key=lambda r: (r.composite_score, -r.distance_meters), reverse=True)
    return evaluated[0]

# Neshan Unwrapping Codec
def unwrap_neshan(s: str, filler: int = 28) -> Any:
    s = s.strip()
    if not s:
        return {}
    if s[0] in '{[':
        return json.loads(s)
    inner = s[filler : len(s) - filler] if filler else s
    at = inner.rfind('@')
    if at < 0 or not inner[at + 1 :].isdigit():
        return {}
    cut, body = int(inner[at + 1 :]), inner[:at]
    raw = body[cut:] + body[:cut]
    return json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)).decode('utf-8'))

print('✅ Matchify core engine and Neshan codec loaded successfully!')

In [ ]:
# Step 3: Configure Paths, auto-detect input shard, and run Neshan matching
import asyncio

# Auto-detect SQLite shard database from Kaggle input directory
input_dbs = sorted(glob.glob('/kaggle/input/**/*.db', recursive=True))
if input_dbs:
    source_db = input_dbs[0]
    print(f'🔍 Auto-detected input shard: {source_db}')
else:
    source_db = 'cafochino_shard_01_of_08_937cafes.db'
    print(f'⚠️ Fallback input database: {source_db}')

output_db = '/kaggle/working/neshan_shard_output.db'

# Async Neshan Runner
async def run_neshan_shard():
    conn_src = sqlite3.connect(source_db)
    cur_src = conn_src.cursor()
    cur_src.execute('SELECT id, name, city, latitude, longitude, phone FROM cafes_cafe WHERE latitude IS NOT NULL AND longitude IS NOT NULL ORDER BY id ASC')
    rows = cur_src.fetchall()
    conn_src.close()

    # Check already processed
    conn_out = sqlite3.connect(output_db)
    cur_out = conn_out.cursor()
    cur_out.execute('''
    CREATE TABLE IF NOT EXISTS places (
        cafe_id INTEGER PRIMARY KEY,
        canonical_id TEXT,
        title TEXT,
        category_name TEXT,
        searched_name TEXT,
        searched_lat REAL,
        searched_lng REAL,
        matched_lat REAL,
        matched_lng REAL,
        address TEXT,
        phone TEXT,
        rating REAL,
        rating_count INTEGER,
        place_url TEXT,
        match_tier TEXT,
        match_rule TEXT,
        name_score REAL,
        distance_meters REAL,
        composite_score REAL,
        phone_matched INTEGER DEFAULT 0,
        rejection_reason TEXT,
        matched_at TEXT
    );
    ''')
    conn_out.commit()
    cur_out.execute('SELECT cafe_id FROM places')
    done_ids = {r[0] for r in cur_out.fetchall()}
    conn_out.close()

    pending = [r for r in rows if r[0] not in done_ids]
    print(f'Total Cafes: {len(rows)} | Already Done: {len(done_ids)} | Pending: {len(pending)}')

    if not pending:
        print('✅ All cafes are already matched!')
        return

    ms = int(time.time() * 1000).to_bytes(6, 'big')
    b = bytearray(ms + secrets.token_bytes(10))
    b[6] = 0x70 | (b[6] & 0x0F)
    b[8] = 0x80 | (b[8] & 0x3F)
    h = b.hex()
    visitor_id = f'web_{h[:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:]}'

    client = httpx.AsyncClient(timeout=10.0, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36', 'Accept': '*/*', 'X-Client-Version': '2038'})

    conn_out = sqlite3.connect(output_db)
    cur_out = conn_out.cursor()

    strict_cnt = 0
    loose_cnt = 0
    unmatched_cnt = 0

    pbar = tqdm(pending, desc='Matching Neshan', unit='cafe')
    try:
        for idx, (cid, name, city, lat, lng, phone) in enumerate(pbar, 1):
            # Dual query search
            clean_name = ' '.join(clean_tokens(name))
            queries = [name.strip()]
            if clean_name and clean_name != name.strip():
                queries.append(clean_name)

            cands = []
            for q in queries:
                body = {'uuid': visitor_id, 'term': q, 'zoom': 15, 'limit': 15, 'filters': {}, 'night': False, 'location': None, 'center': {'x': lng, 'y': lat}, 'boundary': None}
                enc = base64.b64encode(urllib.parse.quote(json.dumps(body, ensure_ascii=False, separators=(',', ':')), safe="-_.!~*'()").encode()).decode()
                try:
                    resp = await client.get('https://neshan.org/maps/pwa-api/neshan-search', params={'body': enc, 'search-in-bound': 'false'}, headers={'Referer': f'https://neshan.org/maps/search/{urllib.parse.quote(q)}'})
                    if resp.status_code == 200:
                        items = unwrap_neshan(resp.text).get('items', [])
                        for it in items:
                            if it.get('isAd'): continue
                            loc = it.get('location') or {}
                            clat, clng = loc.get('y'), loc.get('x')
                            pid = it.get('crowdId')
                            title = (it.get('title') or '').strip()
                            sub = it.get('subtitle')
                            p_call = None
                            for a in it.get('actions') or []:
                                if a.get('type') == 'CALL':
                                    try: p_call = json.loads(a.get('metaData') or '{}').get('phone')
                                    except Exception: pass
                            stars = it.get('rateStars') or {}
                            cands.append(MatchCandidate(candidate_id=pid or f'n_{clat}_{clng}', title=title, latitude=clat, longitude=clng, address=sub, phone=p_call, category=it.get('typeTitle'), url=f'https://neshan.org/maps/places/{pid}' if pid else None, raw_data={'rating': stars.get('rate'), 'votes': stars.get('vote')}))
                except Exception:
                    pass
                if any(c.latitude and math.hypot(c.latitude - lat, (c.longitude or 0) - lng) < 0.005 for c in cands):
                    break
                await asyncio.sleep(0.35)

            match = rank_and_select_best_match(name, lat, lng, phone, cands)

            # Save place
            cand = match.candidate
            now_iso = datetime.now(timezone.utc).isoformat()
            cur_out.execute('''
            INSERT OR REPLACE INTO places (
                cafe_id, canonical_id, title, category_name, searched_name,
                searched_lat, searched_lng, matched_lat, matched_lng, address,
                phone, rating, rating_count, place_url, match_tier, match_rule,
                name_score, distance_meters, composite_score, phone_matched,
                rejection_reason, matched_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                cid,
                cand.candidate_id if cand else None,
                cand.title if cand else None,
                cand.category if cand else None,
                name, lat, lng,
                cand.latitude if cand else None,
                cand.longitude if cand else None,
                cand.address if cand else None,
                cand.phone if cand else None,
                cand.raw_data.get('rating') if cand else None,
                cand.raw_data.get('votes') if cand else None,
                cand.url if cand else None,
                match.match_tier.value,
                match.match_rule,
                match.name_score,
                match.distance_meters if match.distance_meters != float('inf') else None,
                match.composite_score,
                1 if match.phone_matched else 0,
                match.rejection_reason,
                now_iso
            ))

            if match.match_tier == MatchTier.STRICT:
                strict_cnt += 1
            elif match.match_tier == MatchTier.LOOSE:
                loose_cnt += 1
            else:
                unmatched_cnt += 1

            pbar.set_postfix({'Strict': strict_cnt, 'Loose': loose_cnt, 'Unmatched': unmatched_cnt})
            if idx % 25 == 0:
                conn_out.commit()
            await asyncio.sleep(0.35)

    finally:
        conn_out.commit()
        conn_out.close()
        await client.aclose()

    print('✅ Shard execution complete!')

await run_neshan_shard()

In [ ]:
# Step 4: Display Run Statistics & Sample Matches
import sqlite3
import pandas as pd

conn = sqlite3.connect('/kaggle/working/neshan_shard_output.db')
print('=== MATCH TIER BREAKDOWN ===')
df_tiers = pd.read_sql_query('SELECT match_tier, COUNT(*) as count FROM places GROUP BY match_tier', conn)
print(df_tiers)

print('\n=== SAMPLE STRICT MATCHES (~99% CERTAINTY) ===')
df_strict = pd.read_sql_query('SELECT cafe_id, title, searched_name, name_score, distance_meters, phone, address FROM places WHERE match_tier = "strict" LIMIT 5', conn)
print(df_strict)

print('\n=== SAMPLE LOOSE MATCHES (HUMAN REVIEW QUEUE) ===')
df_loose = pd.read_sql_query('SELECT cafe_id, title, searched_name, name_score, distance_meters, phone, address FROM places WHERE match_tier = "loose" LIMIT 5', conn)
print(df_loose)

conn.close()